<a href="https://colab.research.google.com/github/SanthoshSJBIT/GenAI/blob/main/Program_10_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf flask elasticsearch spacy
!python -m spacy download en_core_web_sm
!pip install faiss-cpu


In [ ]:
import fitz  # PyMuPDF
import spacy
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load NLP and embedding models
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 1: Extract text from PDF
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text()
    return text

# Step 2: Chunk text into sections
def chunk_text(text, chunk_size=500):
    sentences = text.split(". ")
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) < chunk_size:
            current_chunk += sentence + ". "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

# Step 3: Create FAISS index
def create_faiss_index(chunks):
    embeddings = model.encode(chunks)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))
    return index, embeddings, chunks

# Step 4: Answer questions
def answer_query(query, index, chunks):
    query_vec = model.encode([query])
    D, I = index.search(np.array(query_vec), k=1)
    return chunks[I[0][0]]


In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_path = next(iter(uploaded))  # Get filename
text = extract_text_from_pdf(pdf_path)
chunks = chunk_text(text)
index, embeddings, chunk_texts = create_faiss_index(chunks)

In [ ]:
query = "What is the punishment for theft?"
response = answer_query(query, index, chunk_texts)
print("Answer:", response)
